In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from pathlib import Path

sys.path.append(os.path.abspath('..'))

from src.hypothesis_tests import (
    chi_square_test, t_test_groups, z_test_proportions,
    interpret_p_value, calculate_claim_frequency,
    calculate_claim_severity, calculate_margin
)

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
# Set correct working directory
project_path = r"C:\Users\stsio\OneDrive\Desktop\insurance-risk-analytics"
os.chdir(project_path)
print(f"Working directory set to: {os.getcwd()}")

# Verify file exists
file_path = Path("data") / "insurance_data_cleaned.csv"
print(f"File exists: {file_path.exists()}")
print(f"Full path: {file_path.absolute()}")

Working directory set to: C:\Users\stsio\OneDrive\Desktop\insurance-risk-analytics
File exists: True
Full path: C:\Users\stsio\OneDrive\Desktop\insurance-risk-analytics\data\insurance_data_cleaned.csv


In [6]:
# Load the cleaned data
df = pd.read_csv(Path("data") / "insurance_data_cleaned.csv")

# Create binary column for claim occurrence
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

print(f"Dataset shape: {df.shape}")
print(f"Policies with claims: {df['HasClaim'].sum():,} ({df['HasClaim'].mean()*100:.2f}%)")
df.head()

C:\Users\stsio\AppData\Local\Temp\ipykernel_4360\1133915941.py:2: DtypeWarning: Columns (0: CapitalOutstanding, 1: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(Path("data") / "insurance_data_cleaned.csv")


Dataset shape: (618174, 53)
Policies with claims: 2,641 (0.43%)


,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,MaritalStatus,Gender,Country,Province,PostalCode,MainCrestaZone,SubCrestaZone,ItemType,mmcode,VehicleType,RegistrationYear,make,Model,Cylinders,cubiccapacity,kilowatts,bodytype,NumberOfDoors,VehicleIntroDate,CustomValueEstimate,AlarmImmobiliser,TrackingDevice,CapitalOutstanding,NewVehicle,WrittenOff,Rebuilt,Converted,CrossBorder,NumberOfVehiclesInFleet,SumInsured,TermFrequency,CalculatedPremiumPerTerm,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims,HasClaim
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0,2597.0,130.0,S/D,4.0,6/2002,119300.0,Yes,No,119300.0,More than 6 months,NaN,NaN,NaN,NaN,NaN,0.01,Monthly,25.0000,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,0
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0,2597.0,130.0,S/D,4.0,6/2002,119300.0,Yes,No,119300.0,More than 6 months,NaN,NaN,NaN,NaN,NaN,0.01,Monthly,25.0000,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,0
2,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0,2597.0,130.0,S/D,4.0,6/2002,119300.0,Yes,No,119300.0,More than 6 months,NaN,NaN,NaN,NaN,NaN,119300.00,Monthly,584.6468,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0,0
3,145247,12827,2015-01-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0,2597.0,130.0,S/D,4.0,6/2002,119300.0,Yes,No,119300.0,More than 6 months,NaN,NaN,NaN,NaN,NaN,500000.00,Monthly,57.5412,No excess,Third Party,Third Party,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,3.256435,0.0,0
4,145247,12827,2015-04-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0,2597.0,130.0,S/D,4.0,6/2002,119300.0,Yes,No,119300.0,More than 6 months,NaN,NaN,NaN,NaN,NaN,500000.00,Monthly,57.5412,No excess,Third Party,Third Party,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,50.474737,0.0,0


In [7]:
print("=" * 60)
print("HYPOTHESIS 1: Risk differences across provinces")
print("=" * 60)
print("H₀: There are no risk differences across provinces")
print("H₁: There ARE significant risk differences across provinces")
print("\nKPI: Claim Frequency (proportion of policies with claims)")

# Calculate claim frequency by province
claim_freq = calculate_claim_frequency(df, 'Province')
print("\nClaim Frequency by Province:")
print(claim_freq.sort_values('claim_frequency', ascending=False))

HYPOTHESIS 1: Risk differences across provinces
H₀: There are no risk differences across provinces
H₁: There ARE significant risk differences across provinces

KPI: Claim Frequency (proportion of policies with claims)

Claim Frequency by Province:
               total_policies  policies_with_claims  claim_frequency
Province                                                            
Gauteng                240781                  1243         0.005162
KwaZulu-Natal          111896                   453         0.004048
Mpumalanga              31663                   125         0.003948
North West              89799                   334         0.003719
Western Cape            96757                   356         0.003679
Limpopo                 18009                    66         0.003665
Eastern Cape            19694                    47         0.002387
Northern Cape            3643                     8         0.002196
Free State               5932                     9         0.

In [8]:
# Chi-square test for provinces
chi2, p, dof, expected = chi_square_test(df, 'Province', 'HasClaim')

print("=" * 60)
print("CHI-SQUARE TEST RESULTS")
print("=" * 60)
print(f"Chi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p:.6f}")

decision, interpretation = interpret_p_value(p)
print(f"\nDecision: {decision}")
print(f"Interpretation: {interpretation}")

if p < 0.05:
    print("\n REJECT H₀: There ARE significant risk differences across provinces")
    print("\n Business Insight:")
    print("   - Gauteng has the HIGHEST claim frequency")
    print("   - Northern Cape has the LOWEST claim frequency")
    print("   → Recommendation: Adjust premiums based on province risk profile")
else:
    print("\n FAIL TO REJECT H₀: No significant risk differences across provinces")

CHI-SQUARE TEST RESULTS
Chi-square statistic: 93.6952
Degrees of freedom: 8
P-value: 0.000000

Decision: REJECT H₀
Interpretation: p = 0.0000 < 0.05 - Statistically significant

 REJECT H₀: There ARE significant risk differences across provinces

 Business Insight:
   - Gauteng has the HIGHEST claim frequency
   - Northern Cape has the LOWEST claim frequency
   → Recommendation: Adjust premiums based on province risk profile


In [9]:
print("=" * 60)
print("HYPOTHESIS 2: Risk differences between zip codes")
print("=" * 60)
print("H₀: There are no risk differences between zip codes")
print("H₁: There ARE significant risk differences between zip codes")
print("\nKPI: Claim Severity (average claim amount when claim occurs)")

# Get top 2 zip codes by number of policies for comparison
zip_counts = df['PostalCode'].value_counts()
top_zips = zip_counts.head(2).index.tolist()
print(f"\nComparing zip codes: {top_zips[0]} vs {top_zips[1]}")
print(f"Sample sizes: {top_zips[0]}: {zip_counts[top_zips[0]]}, {top_zips[1]}: {zip_counts[top_zips[1]]}")

# Calculate severity by zip code
severity = calculate_claim_severity(df, 'PostalCode')
print(f"\nClaim Severity (average claim amount):")
print(f"  Zip {top_zips[0]}: R{severity[top_zips[0]]:.2f}")
print(f"  Zip {top_zips[1]}: R{severity[top_zips[1]]:.2f}")

HYPOTHESIS 2: Risk differences between zip codes
H₀: There are no risk differences between zip codes
H₁: There ARE significant risk differences between zip codes

KPI: Claim Severity (average claim amount when claim occurs)

Comparing zip codes: 2000 vs 122
Sample sizes: 2000: 90934, 122: 27898

Claim Severity (average claim amount):
  Zip 2000: R19235.77
  Zip 122: R17431.77


In [10]:
# T-test for claim severity between top 2 zip codes
t_stat, p_value_zip = t_test_groups(df, 'PostalCode', top_zips[0], top_zips[1], 'TotalClaims')

print("=" * 60)
print("T-TEST RESULTS")
print("=" * 60)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value_zip:.6f}")

decision, interpretation = interpret_p_value(p_value_zip)
print(f"\nDecision: {decision}")
print(f"Interpretation: {interpretation}")

if p_value_zip < 0.05:
    print("\n✅ REJECT H₀: There ARE significant risk differences between zip codes")
    print("\n📊 Business Insight:")
    print(f"   - Zip {top_zips[0]} has average claim of R{severity[top_zips[0]]:.2f}")
    print(f"   - Zip {top_zips[1]} has average claim of R{severity[top_zips[1]]:.2f}")
    print("   → Recommendation: Implement zip-code based pricing adjustments")
else:
    print("\n❌ FAIL TO REJECT H₀: No significant risk differences between selected zip codes")

T-TEST RESULTS
T-statistic: -0.9147
P-value: 0.360368

Decision: FAIL TO REJECT H₀
Interpretation: p = 0.3604 >= 0.05 - Not statistically significant

❌ FAIL TO REJECT H₀: No significant risk differences between selected zip codes


In [12]:
print("=" * 60)
print("HYPOTHESIS 3: Margin differences between zip codes")
print("=" * 60)
print("H₀: There is no significant margin difference between zip codes")
print("H₁: There IS a significant margin difference between zip codes")
print("\nKPI: Margin (TotalPremium - TotalClaims)")

# Calculate margin
df['Margin'] = df['TotalPremium'] - df['TotalClaims']

# Calculate margin by zip code
margin_by_zip = calculate_margin(df, 'PostalCode')
print(f"\nMargin (average profit per policy):")
print(f"  Zip {top_zips[0]}: R{margin_by_zip[top_zips[0]]:.2f}")
print(f"  Zip {top_zips[1]}: R{margin_by_zip[top_zips[1]]:.2f}")

# T-test for margin
t_stat_margin, p_value_margin = t_test_groups(df, 'PostalCode', top_zips[0], top_zips[1], 'Margin')

print("\n" + "=" * 60)
print("T-TEST RESULTS (Margin)")
print("=" * 60)
print(f"T-statistic: {t_stat_margin:.4f}")
print(f"P-value: {p_value_margin:.6f}")

decision, interpretation = interpret_p_value(p_value_margin)
print(f"\nDecision: {decision}")

if p_value_margin < 0.05:
    print("\nREJECT H₀: There IS a significant margin difference between zip codes")
    print("\n Business Insight:")
    print(f"   - Zip {top_zips[0]} has average margin of R{margin_by_zip[top_zips[0]]:.2f}")
    print(f"   - Zip {top_zips[1]} has average margin of R{margin_by_zip[top_zips[1]]:.2f}")
    print("   → Recommendation: Reallocate marketing spend to higher-margin zip codes")
else:
    print("\n FAIL TO REJECT H₀: No significant margin difference between selected zip codes")

HYPOTHESIS 3: Margin differences between zip codes
H₀: There is no significant margin difference between zip codes
H₁: There IS a significant margin difference between zip codes

KPI: Margin (TotalPremium - TotalClaims)

Margin (average profit per policy):
  Zip 2000: R-8.94
  Zip 122: R-22.30

T-TEST RESULTS (Margin)
T-statistic: 0.6424
P-value: 0.520588

Decision: FAIL TO REJECT H₀

 FAIL TO REJECT H₀: No significant margin difference between selected zip codes


In [14]:
print("=" * 60)
print("HYPOTHESIS 4: Risk differences between Women and Men")
print("=" * 60)
print("H₀: There is no significant risk difference between Women and Men")
print("H₁: There IS a significant risk difference between Women and Men")
print("\nKPI: Claim Frequency (proportion of policies with claims)")

# Filter for Male and Female only
gender_df = df[df['Gender'].isin(['Male', 'Female'])].copy()

# Calculate claim frequency by gender
gender_freq = gender_df.groupby('Gender')['HasClaim'].mean()
print(f"\nClaim Frequency by Gender:")
print(f"  Male: {gender_freq['Male']*100:.2f}%")
print(f"  Female: {gender_freq['Female']*100:.2f}%")

# Get counts for z-test
male_claims = gender_df[gender_df['Gender'] == 'Male']['HasClaim'].sum()
male_total = len(gender_df[gender_df['Gender'] == 'Male'])
female_claims = gender_df[gender_df['Gender'] == 'Female']['HasClaim'].sum()
female_total = len(gender_df[gender_df['Gender'] == 'Female'])

# Z-test for proportions
z_stat, p_value_gender = z_test_proportions(male_claims, male_total, female_claims, female_total)

print("\n" + "=" * 60)
print("Z-TEST RESULTS (Proportions)")
print("=" * 60)
print(f"Male: {male_claims}/{male_total} ({male_claims/male_total*100:.2f}%)")
print(f"Female: {female_claims}/{female_total} ({female_claims/female_total*100:.2f}%)")
print(f"\nZ-statistic: {z_stat:.4f}")
print(f"P-value: {p_value_gender:.6f}")

decision, interpretation = interpret_p_value(p_value_gender)
print(f"\nDecision: {decision}")

if p_value_gender < 0.05:
    print("\n REJECT H₀: There IS a significant risk difference between genders")
    print("\n Business Insight:")
    print(f"   - Male claim frequency: {male_claims/male_total*100:.2f}%")
    print(f"   - Female claim frequency: {female_claims/female_total*100:.2f}%")
    print("   → Recommendation: Consider gender-specific premium adjustments")
else:
    print("\nFAIL TO REJECT H₀: No significant risk difference between genders")
    print("\nBusiness Insight:")
    print("   - Male and female drivers show similar claim patterns")
    print("   → Recommendation: Gender should NOT be a factor in premium pricing")

HYPOTHESIS 4: Risk differences between Women and Men
H₀: There is no significant risk difference between Women and Men
H₁: There IS a significant risk difference between Women and Men

KPI: Claim Frequency (proportion of policies with claims)

Claim Frequency by Gender:
  Male: 0.45%
  Female: 0.38%

Z-TEST RESULTS (Proportions)
Male: 85/19083 (0.45%)
Female: 13/3404 (0.38%)

Z-statistic: 0.5183
P-value: 0.604269

Decision: FAIL TO REJECT H₀

FAIL TO REJECT H₀: No significant risk difference between genders

Business Insight:
   - Male and female drivers show similar claim patterns
   → Recommendation: Gender should NOT be a factor in premium pricing


In [15]:
results = []

# Hypothesis 1 (using p from Cell 5)
results.append({
    'Hypothesis': 'H₁: Risk differences across provinces',
    'KPI': 'Claim Frequency',
    'Test': 'Chi-square',
    'P-value': f'{p:.6f}',
    'Decision': 'REJECT H₀' if p < 0.05 else 'FAIL TO REJECT',
    'Business Recommendation': 'Adjust premiums based on province risk profile'
})

# Hypothesis 2
results.append({
    'Hypothesis': 'H₂: Risk differences between zip codes',
    'KPI': 'Claim Severity',
    'Test': 'T-test',
    'P-value': f'{p_value_zip:.6f}',
    'Decision': 'REJECT H₀' if p_value_zip < 0.05 else 'FAIL TO REJECT',
    'Business Recommendation': 'Implement zip-code based pricing adjustments' if p_value_zip < 0.05 else 'No adjustment needed'
})

# Hypothesis 3
results.append({
    'Hypothesis': 'H₃: Margin differences between zip codes',
    'KPI': 'Margin',
    'Test': 'T-test',
    'P-value': f'{p_value_margin:.6f}',
    'Decision': 'REJECT H₀' if p_value_margin < 0.05 else 'FAIL TO REJECT',
    'Business Recommendation': 'Reallocate marketing to higher-margin zip codes' if p_value_margin < 0.05 else 'Current allocation is optimal'
})

# Hypothesis 4
results.append({
    'Hypothesis': 'H₄: Risk differences between genders',
    'KPI': 'Claim Frequency',
    'Test': 'Z-test',
    'P-value': f'{p_value_gender:.6f}',
    'Decision': 'REJECT H₀' if p_value_gender < 0.05 else 'FAIL TO REJECT',
    'Business Recommendation': 'Gender should NOT be a pricing factor' if p_value_gender >= 0.05 else 'Consider gender-based adjustments'
})

results_df = pd.DataFrame(results)

print("=" * 80)
print("A/B HYPOTHESIS TESTING - RESULTS SUMMARY")
print("=" * 80)
print(results_df.to_string(index=False))

# Save to CSV
results_df.to_csv('reports/hypothesis_testing_results.csv', index=False)
print("\n✓ Results saved to reports/hypothesis_testing_results.csv")

A/B HYPOTHESIS TESTING - RESULTS SUMMARY
                              Hypothesis             KPI       Test  P-value       Decision                        Business Recommendation
   H₁: Risk differences across provinces Claim Frequency Chi-square 0.000000      REJECT H₀ Adjust premiums based on province risk profile
  H₂: Risk differences between zip codes  Claim Severity     T-test 0.360368 FAIL TO REJECT                           No adjustment needed
H₃: Margin differences between zip codes          Margin     T-test 0.520588 FAIL TO REJECT                  Current allocation is optimal
    H₄: Risk differences between genders Claim Frequency     Z-test 0.604269 FAIL TO REJECT          Gender should NOT be a pricing factor

✓ Results saved to reports/hypothesis_testing_results.csv
